**Linda Zier**

**ST 554 ~ Final Project**

**04/30/2026**

# Goal

The goal of this project was to use Spark to fit a machine learning model and apply it to streaming data. All project files were housed in a GitHub repository, including a Jupyter notebook and a Python producer script. In the notebook, I fit an elastic net regression model using PySpark's MLlib module to predict power consumption in Zone 3 of Tetouan City from weather and time variables. I then simulated a data stream by writing a separate Python script that periodically wrote batches of data to a monitored folder. As data arrived in the stream, I used the fitted model to generate predictions and wrote the results out to the console.

### Data

The data used in this project is modified from the UCI Machine Learning Repository and is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The dataset contains measurements from Tetouan City relating power consumption across three zones to factors such as temperature, humidity, wind speed, diffuse solar flows, and time of day. I used this dataset to fit my model, treating Power Zone 3 consumption as the response variable. A separate streaming dataset (power_streaming_data.csv) was used to simulate incoming data. My producer script repeatedly sampled from this file and wrote batches to a monitored folder where the fitted model generated predictions on the arriving data.

# Fitting the Model

I created a Jupyter notebook for the model fitting part and the streaming part below. I read the data into a standard pandas data frame using the pd.read_csv() function and converted this to a spark data frame.

In [4]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 14:22:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Creating the Pipeline

I fit an elastic net model using cross validation with the steps below. The transformations used an MLlib function that was put into a pipeline.

*   I used an SQL transformer to cast the hour variable as DoubleType.
*   I binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).
*   The month column was one-hot encoded.
*   I ran a Principle Component Analysis (PCA) on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. I did this by:

        - First using a VectorAssembler() to place these variables in a single column for use with the PCA() estimator  
        
        - Then applying a PCA transformer for use in our pipeline using two principle components.
        
        
*   I renamed our response variable Power_Zone_3 as label.

*   I used VectorAssembler() to put my predictors into a features vector. The predictors were:

    – Two fitted PCA features
    
    – Binary Hour variable
    
    – Power_Zone_1
    
    – Power_Zone_2
    
    – Month indicator variables
    
I then built my pipeline for the transformations.
    


In [5]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

TRANSFORMATIONS COMPLETE


In [6]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

PIPELINE COMPLETE


### Fitting an Elastic Net Model

Next I used the CrossValidator and LinearRegression functions to fit an elastic net model. I searched over multiple combinations of regularization parameters, which control the strength of the penalty, and elastic net parameters, which control the mix between Lasso and Ridge. The model was fit using 5-fold cross validation with root mean square error (RMSE) as the criteria. I trained 5 separate models (one per fold) and averaged their RMSEs to evaluate each parameter combination. I then reported the optimal tuning parameter values and the CV error, which is the RMSE from the best model.


In [7]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regularization parameter: ", cvModel.bestModel.getRegParam())
print("Optimal elastic net parameter: ", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE: ", min(cvModel.avgMetrics))


26/04/30 14:22:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 14:22:44 WARN Instrumentation: [720ae8ce] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:22:45 WARN Instrumentation: [720ae8ce] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 14:22:47 WARN Instrumentation: [103a795b] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:22:47 WARN Instrumentation: [103a795b] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 14:22:47 WARN Instrumentation: [938d16f3] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 14:22:47 WARN Instrumentation: [938d16f3] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regularization parameter:  0.25
Optimal elastic net parameter:  0.05
CV RMSE:  2148.0523195543183


### Training Set Evaluation
I reported the training set RMSE using our fitted model as a transformer and
evaluating on the entire training set. I then created a residual column (label - prediction) and printed the data frame with these residuals. I also printed a summary table as a sanity check to ensure the mean was near zero and the standard deviation was reasonable.

In [8]:
# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
print("Training set residuals and summary:")
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.0974532567648
Training set residuals and summary:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20879.94937560797|-638.9855156079684|
|20131.08434| 18659.74766483353|1471.3366751664726|
|19668.43373| 18204.27001392659|  1464.16371607341|
|18899.27711| 17590.20919357923|1309.0679164207686|
|18442.40964| 16996.86206723391|1445.5475727660923|
|18130.12048| 16517.26525889357|1612.8552211064307|
|17945.06024|16092.847067605384| 1852.213172394615|
|17459.27711|15722.304216985289|1736.9728930147103|
|17025.54217|15270.671568238638|1754.8706017613622|
|16794.21687|14937.980553619505|1856.2363163804948|
|16638.07229|14652.098414968634|1985.9738750313663|
|16395.18072| 14414.62351266616|1980.5572073338408|
|16117.59036|14082.540395195432| 2035.049964804568|
| 15822.6506| 13624.55837106434|2198.0922289356604|
|15672.28916| 13450.06174223221|  2222.2274177

# Handling Streaming Data
I downloaded the streaming source file from https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in my final_project/data directory.  This file served as my source for random sampling in my producer script.
### Reading a Stream
I read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data to serve as the monitored directory. The schema was set to that of the original data since that is what my incoming data would have the same structure and a header was assumed to be present.

In [9]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream with a header
streamDF = spark.readStream.schema(stream_schema).option("header", True)\
           .csv("streaming_data")

#showing current working directory
#import os
#os.getcwd()

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])



### Transform/Aggregation Step

In this code block I used the fitted model as a transformer to obtain predictions from the incoming data stream. First I created a residual column (label - prediction) as done in the previous section, returning only the label, prediction, and residual columns. Then, using a second transformation on the original stream, I renamed the response variable Power_Zone_3 to label, keeping all other columns intact. Finally I joined the two transformations together on the label variable using an inner join.

In [10]:
# ---Transformation 1:---

# apply transformer to the data stream
streamPredictions = cvModel.transform(fittedPipeline.transform(streamDF))

#add residual column = label - predictions
streamResiduals=streamPredictions.withColumn("residual",col("label")- col("prediction")) \
                                  .select("label","prediction", "residual")
                                              
# ---Transformation 2:---

# rename response to
streamLabeled=sql_label.transform(streamDF)

# --- Inner Join ---
streamJoined = streamResiduals.join(streamLabeled, on="label", how="inner")



### Writing Step
I wrote the joined stream to the console using append output mode, which outputs only newly arriving rows with each batch. I started the query and allowed it to run before terminating it with query.stop().

In [11]:
# Write stream to console in append mode and start query
#query = streamJoined.writeStream.outputMode("append").format("console")

query = streamJoined.writeStream \
                    .outputMode("append") \
                    .format("console") \
                    .start()


26/04/30 14:26:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7670dfa6-dfe4-484d-a1f8-2915d9de0618. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 14:26:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [13]:
# stop querying
query.stop()

### the other commands below were used when repeatedly testing
#for s in spark.streams.active:
#    s.stop()
    
#for f in os.listdir("streaming_data"):
#    os.remove(f"streaming_data/{f}")